# Detecção de duplicatas
**Fernando Amorim Pontes Neto**

Cada par aparece uma única vez; a diagonal é excluída. O critério é similaridade estritamente maior que o limiar.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
from ouvidoria_core import *
base = carregar_base('manifestacoes.json')
assert len(base) == 40, 'A análise do desafio requer exatamente 40 registros.'
textos = base.texto.tolist()
ids = base.id.tolist()
saida = Path('resultados'); saida.mkdir(exist_ok=True)
print(f'Base: {len(base)} registros; textos >500 caracteres: {(base.texto.str.len()>500).sum()}')
if Path('PROVENIENCIA.json').exists():
    display(Markdown('**Proveniência:** ' + json.loads(Path('PROVENIENCIA.json').read_text())['descricao']))
display(base.head())

/home/user/ouvidoria/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base: 40 registros; textos >500 caracteres: 5


**Proveniência:** Base demonstrativa sintética criada para o exercício, com autorização do usuário. Não corresponde a manifestações reais nem à base original do professor. Os resultados e o gabarito se aplicam apenas a estes 40 exemplos.

,id,data,categoria_oficial,texto
0,M001,2026-09-01,infraestrutura,"A calçada da Rua das Flores, perto do número 1..."
1,M002,2026-09-01,educação,A Escola Municipal Esperança está sem professo...
2,M003,2026-09-02,infraestrutura,"Há um buraco enorme na Av. Brasil, a avenida p..."
3,M004,2026-09-02,segurança,Moradores relatam furtos de bicicletas junto a...
4,M005,2026-09-03,meio ambiente,Uma pilha de pneus usados foi abandonada no te...


In [2]:
e=codificar(textos,MODELOS[0]); matriz=cosine_similarity(e)
pares=pares_da_matriz(matriz,0.85,ids)
display(pd.DataFrame(pares,columns=['id_a','id_b','similaridade']))
pd.DataFrame(pares,columns=['id_a','id_b','similaridade']).to_csv(saida/'duplicatas.csv',index=False)
# A função solicitada retorna índices da lista; a versão com IDs utiliza a mesma matriz.
assert len(detectar_duplicatas(textos,limiar=0.85)) == len(pares)
fig,ax=plt.subplots(figsize=(11,9))
im=ax.imshow(matriz,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(40),ids,rotation=90,fontsize=7); ax.set_yticks(range(40),ids,fontsize=7)
fig.colorbar(im,ax=ax,label='Cosseno'); fig.tight_layout()
fig.savefig(saida/'heatmap.png',dpi=150); plt.show()
pd.DataFrame(matriz,index=ids,columns=ids).to_csv(saida/'matriz_similaridade.csv')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3600.03it/s]

,id_a,id_b,similaridade
0,M008,M022,0.921487
1,M005,M026,0.882144


## Escolha do limiar
0,85 é o ponto de partida conservador solicitado, não uma garantia de duplicidade. O percentil 90 é calculado apenas nos 780 pares únicos, excluindo a diagonal. Esse percentil admite aproximadamente 10% dos pares por construção, mesmo quando não existem duplicatas; não equivale aos 15% de registros mencionados no enunciado. A inspeção abaixo compara vários limiares. Ajustar e medir no mesmo corpus é exploratório: uma validação real deve separar eventos na calibração e no teste para evitar vazamento.

In [3]:
valores=matriz[np.triu_indices(len(ids),k=1)]
p90=float(np.percentile(valores,90))
print('Pares únicos:',len(valores),'Percentil 90:',round(p90,4))
limiares=sorted(set([0.65,0.70,0.75,0.80,0.85,0.90,0.95,p90]))
gabarito=Path('duplicatas_reais.json')
verdade=json.loads(gabarito.read_text()) if gabarito.exists() else None
avaliacoes=[]
for limiar in limiares:
    previstos=pares_da_matriz(matriz,limiar,ids)
    r={'limiar':limiar,'pares_detectados':len(previstos)}
    if verdade is not None:
        av=avaliar_pares(previstos,verdade,ids)
        r.update({k:av[k] for k in ['precision','recall','F1']})
    avaliacoes.append(r)
display(pd.DataFrame(avaliacoes).round(4))
resumo={'limiar':0.85,'percentil_90':p90,'pares':pares,'sensibilidade':avaliacoes}
if verdade is None:
    display(Markdown('**Sem gabarito exaustivo, não é possível calcular falsos positivos, falsos negativos, precisão ou recall.** Forneça duplicatas_reais.json como lista de pares de IDs. Pares ilustrativos do enunciado não bastam como gabarito.'))
else:
    av=avaliar_pares(pares,verdade,ids); resumo['avaliacao']=av
    display(pd.DataFrame([{k:av[k] for k in ['precision','recall','F1']}]))
    for tipo in ['TP','FP','FN']:
        print('\n',tipo,':',av[tipo])
        for a,b in av[tipo]:
            display(base[base.id.isin([a,b])][['id','data','texto']])
    display(Markdown(f"Com 0,85, foram encontrados **{len(av['TP'])} verdadeiros positivos**, **{len(av['FP'])} falsos positivos** e **{len(av['FN'])} falsos negativos**. Falsos positivos devem ser revistos quanto a locais e eventos distintos; falsos negativos indicam paráfrases pouco reconhecidas ou contexto adicional que reduz o cosseno. Alterar o limiar troca precisão por cobertura e não resolve sozinho a distinção entre tema e ocorrência."))
(saida/'duplicatas_resumo.json').write_text(json.dumps(resumo,ensure_ascii=False,indent=2))

Pares únicos: 780 Percentil 90: 0.502


,limiar,pares_detectados,precision,recall,F1
0,0.502,78,0.0385,1.0000,0.0741
1,0.650,12,0.2500,1.0000,0.4000
2,0.700,7,0.4286,1.0000,0.6000
3,0.750,5,0.6000,1.0000,0.7500
4,0.800,2,1.0000,0.6667,0.8000
5,0.850,2,1.0000,0.6667,0.8000
6,0.900,1,1.0000,0.3333,0.5000
7,0.950,0,0.0000,0.0000,0.0000


,precision,recall,F1
0,1.0,0.666667,0.8



 TP : [('M005', 'M026'), ('M008', 'M022')]


,id,data,texto
4,M005,2026-09-03,Uma pilha de pneus usados foi abandonada no te...
25,M026,2026-09-03,"No lote vazio da Rua do Sol, junto ao imóvel 9..."


,id,data,texto
7,M008,2026-09-04,"O posto de saúde São Lucas, na Rua das Acácias..."
21,M022,2026-09-04,Está faltando atendimento no PSF São Lucas da ...



 FP : []

 FN : [('M003', 'M017')]


,id,data,texto
2,M003,2026-09-02,"Há um buraco enorme na Av. Brasil, a avenida p..."
16,M017,2026-09-02,O asfalto está todo esburacado na avenida prin...


Com 0,85, foram encontrados **2 verdadeiros positivos**, **0 falsos positivos** e **1 falsos negativos**. Falsos positivos devem ser revistos quanto a locais e eventos distintos; falsos negativos indicam paráfrases pouco reconhecidas ou contexto adicional que reduz o cosseno. Alterar o limiar troca precisão por cobertura e não resolve sozinho a distinção entre tema e ocorrência.

1705

## Decisão operacional
Os pares detectados são candidatos para revisão humana, nunca registros a excluir automaticamente. Data próxima, endereço compatível e descrição do mesmo evento fortalecem a evidência; categoria igual não basta. O gabarito deve ser revisado por quem conhece os casos e sua proveniência deve acompanhar a análise.